In [ ]:
import sys
from pathlib import Path

current_dir = Path.cwd()

def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'pyproject.toml').exists() and (p / 'bcosgnn').is_dir():
            return p
    raise RuntimeError('Could not locate repo root (pyproject.toml + bcosgnn/).')

project_root = find_repo_root(current_dir)

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

print(f"Repo root added: {project_root}")


In [ ]:
import random
import numpy as np
import torch
import polars as pl
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from torch.nn import CrossEntropyLoss
from torch_geometric.data import InMemoryDataset
from torch_geometric.loader import DataLoader
from tqdm import tqdm

from bcosgnn.explain_edge_attr import explain as explain_edge_attr
from bcosgnn.evaluation import (
    evaluate_auroc_edge,
    evaluate_jaccard_edge,
    evaluate_gnnexplainer_jaccard_edge,
    evaluate_gnnexplainer_auroc_edge,
)
from bcosgnn.sanitized_models import BCosGINE, BcosGINEConv, ReadoutThenAgg


## Dataset Loading


In [ ]:
class ZincProcessedDataset(InMemoryDataset):
    """Thin wrapper around a pre-processed ZINC InMemoryDataset stored in data.pt."""

    def __init__(self, root, transform=None, pre_transform=None):
        super().__init__(root, transform, pre_transform)
        data_path = Path(self.processed_dir) / "data.pt"
        print(f"Loading data from: {data_path.resolve()}")
        try:
            self.data, self.slices = torch.load(data_path, weights_only=False)
        except TypeError:
            self.data, self.slices = torch.load(data_path)

    @property
    def raw_file_names(self):  return []
    @property
    def processed_file_names(self): return ['data.pt']
    def download(self): pass
    def process(self):  pass


# ── Path resolution ───────────────────────────────────────────────────────────
CANDIDATE_PATHS = [
    project_root / "shaique_updates/codes/multi_class_Zinc/zinc_di_halo_benzene_data",
    Path("/Users/shaique/Desktop/BioInf_IMP/NMM_group/ICML/bcos_gnn/bcosgnn/shaique_updates/codes/multi_class_Zinc/zinc_di_halo_benzene_data"),
]
DATASET_PATH = next((str(p) for p in CANDIDATE_PATHS if p.exists()), None)
if DATASET_PATH is None:
    raise FileNotFoundError(
        "Cannot find zinc_di_halo_benzene_data. Set DATASET_PATH manually."
    )

dataset = ZincProcessedDataset(root=DATASET_PATH)
print(f"\nDataset: {len(dataset)} graphs")
print(f"Node features : {dataset.num_node_features}")
print(f"Edge features : {dataset.num_edge_features}")
print(f"Classes       : {dataset.num_classes}")

# ── Quick stats ───────────────────────────────────────────────────────────────
n        = len(dataset)
avg_nodes = sum(dataset[i].num_nodes for i in range(n)) / n
avg_edges = sum(dataset[i].num_edges for i in range(n)) / n
y_counts  = {}
for i in range(n):
    y = int(dataset[i].y.item())
    y_counts[y] = y_counts.get(y, 0) + 1

print(f"\nAvg nodes/graph : {avg_nodes:.1f}")
print(f"Avg edges/graph : {avg_edges:.1f}")
print("Class distribution:")
CLASS_LABELS = [
    "di_chloro_ortho", "di_chloro_meta", "di_chloro_para",
    "di_fluoro_ortho", "di_fluoro_meta", "di_fluoro_para",
    "di_bromo_ortho",  "di_bromo_meta",  "di_bromo_para",
]
for c in sorted(y_counts):
    label = CLASS_LABELS[c] if c < len(CLASS_LABELS) else str(c)
    print(f"  Class {c} ({label}): {y_counts[c]} graphs")


## Train B-COS GINE (Sanitized)

Uses the modular `BCosGINE` from `bcosgnn.sanitized_models` — a fully B-COS linear
GINE model (no dropout, no standard non-linearities) that supports **exact intrinsic
explanations** for both node and edge features via `explain_edge_attr`.

Architecture per seed:
- `lin_node` + `lin_edge`: 9-dim / edge-dim → 128 (B-COS linear)
- 4 × `BcosGINEConv([128, 128])` — additive message x_j + e_ij, then B-COS MLP
- `ReadoutThenAgg(128 → 128 → num_classes)` — node projection, then sum pool


In [ ]:
from torch.optim.lr_scheduler import ReduceLROnPlateau

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# ── Reproducibility helpers ───────────────────────────────────────────────────

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def split_train_val_test(dataset, seed, test_size=0.2, val_size=0.1):
    """Stratified train / val / test split.  Returns index arrays."""
    labels = np.array([int(dataset[i].y.item()) for i in range(len(dataset))])
    indices = np.arange(len(dataset))
    train_val_idx, test_idx = train_test_split(
        indices, test_size=test_size, random_state=seed, stratify=labels
    )
    tv_labels = labels[train_val_idx]
    val_ratio = val_size / (1.0 - test_size)
    train_idx, val_idx = train_test_split(
        train_val_idx, test_size=val_ratio, random_state=seed, stratify=tv_labels
    )
    return train_idx, val_idx, test_idx


def make_loader(indices, dataset, batch_size=64, shuffle=False):
    subset = dataset.index_select(torch.tensor(indices))
    return DataLoader(subset, batch_size=batch_size, shuffle=shuffle)


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = total_correct = total_graphs = 0
    for batch in loader:
        batch = batch.to(DEVICE)
        logits = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
        loss   = criterion(logits, batch.y.view(-1).long())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        pred = logits.argmax(dim=-1)
        total_loss    += float(loss.item()) * batch.num_graphs
        total_correct += int((pred == batch.y.view(-1).long()).sum().item())
        total_graphs  += int(batch.num_graphs)
    return total_loss / total_graphs, total_correct / total_graphs


@torch.inference_mode()
def evaluate_loader(model, loader, criterion):
    model.eval()
    total_loss = total_correct = total_graphs = 0
    for batch in loader:
        batch = batch.to(DEVICE)
        logits = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
        loss   = criterion(logits, batch.y.view(-1).long())
        pred   = logits.argmax(dim=-1)
        total_loss    += float(loss.item()) * batch.num_graphs
        total_correct += int((pred == batch.y.view(-1).long()).sum().item())
        total_graphs  += int(batch.num_graphs)
    return {"loss": total_loss / total_graphs, "acc": total_correct / total_graphs}


class EvalModelAdapter(torch.nn.Module):
    """Wraps BCosGINE so explain_edge_attr and PyG Explainer can call it uniformly.

    Converts the strict positional signature ``forward(x, edge_index, edge_attr, batch)``
    into ``forward(x, edge_index, edge_attr=None, batch=None)`` with auto-batch for
    single graphs.  No logic is changed — edge_attr is always passed through.
    """
    def __init__(self, base_model: torch.nn.Module):
        super().__init__()
        self.base_model = base_model

    def forward(self, x, edge_index, edge_attr=None, batch=None):
        if batch is None:
            batch = torch.zeros(x.size(0), dtype=torch.long, device=x.device)
        return self.base_model(x, edge_index, edge_attr, batch)


# ── Hyperparameters ───────────────────────────────────────────────────────────
node_size   = dataset.num_node_features
edge_size   = dataset.num_edge_features
num_classes = dataset.num_classes

SEEDS      = [0, 1, 2, 3, 4]
EPOCHS     = 200
BATCH_SIZE = 64
LR         = 1e-3
hidden_dim = 128
b          = 2.0

seed_results      = []
best_result       = None
best_result_score = -float("inf")

# ── Multi-seed training loop ──────────────────────────────────────────────────
for seed in SEEDS:
    print("=" * 80)
    print(f"Seed {seed}")
    print("=" * 80)
    set_seed(seed)

    train_idx, val_idx, test_idx = split_train_val_test(dataset, seed=seed)
    train_loader = make_loader(train_idx, dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = make_loader(val_idx,   dataset, batch_size=BATCH_SIZE, shuffle=False)
    # Build list for per-graph explanation evaluation (avoid DataLoader overhead)
    test_data = [dataset[int(i)] for i in test_idx]

    readout = ReadoutThenAgg(
        in_channels=hidden_dim,
        hidden_channels=[hidden_dim],   # hidden_dim → hidden_dim → num_classes
        out_channels=num_classes,
        b=b,
        max_out=1,
        agg="sum",
    )

    model = BCosGINE(
        node_size=node_size,
        edge_size=edge_size,
        hidden_channels=[hidden_dim, hidden_dim],   # conv MLP: hidden_dim → hidden_dim
        num_convs=4,
        readout=readout,
        b=b,
        max_out=1,
    ).to(DEVICE)

    criterion = CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=10, min_lr=1e-6)

    best_state    = None
    best_val_loss = float("inf")
    best_epoch    = -1

    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        val_metrics     = evaluate_loader(model, val_loader,   criterion)
        scheduler.step(val_metrics["loss"])

        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            best_epoch    = epoch
            best_state    = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if epoch % 10 == 0 or epoch == 1:
            print(
                f"Epoch {epoch:03d} | train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} "
                f"| val_loss={val_metrics['loss']:.4f} val_acc={val_metrics['acc']:.4f} "
                f"| lr={optimizer.param_groups[0]['lr']:.2e}"
            )

    model.load_state_dict(best_state)
    eval_model = EvalModelAdapter(model).to(DEVICE)

    test_loader     = make_loader(test_idx, dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_metrics    = evaluate_loader(model, test_loader, criterion)
    test_jaccard    = float(evaluate_jaccard_edge(eval_model, test_data))
    test_expl_auroc = float(evaluate_auroc_edge(eval_model,   test_data))

    result = {
        "seed":            seed,
        "best_epoch":      best_epoch,
        "best_val_loss":   float(best_val_loss),
        "test_loss":       float(test_metrics["loss"]),
        "test_acc":        float(test_metrics["acc"]),
        "test_expl_auroc": test_expl_auroc,
        "test_jaccard":    test_jaccard,
        "model":           model,
        "eval_model":      eval_model,
        "test_dataset":    test_data,
    }
    seed_results.append(result)
    print(
        f"Seed {seed} | best_epoch={best_epoch} | test_acc={result['test_acc']:.4f} "
        f"| test_expl_auroc={result['test_expl_auroc']:.4f} "
        f"| test_jaccard={result['test_jaccard']:.4f}"
    )

    rank_score = (
        np.nan_to_num(test_expl_auroc, nan=-1e9)
        + np.nan_to_num(test_jaccard,  nan=-1e9)
    )
    if best_result is None or rank_score > best_result_score:
        best_result_score = rank_score
        best_result = result

# ── Summary ───────────────────────────────────────────────────────────────────
metrics_keys = ["test_loss", "test_acc", "test_expl_auroc", "test_jaccard"]
summary = {}
for key in metrics_keys:
    vals = np.array([r[key] for r in seed_results], dtype=float)
    summary[f"{key}_mean"] = float(np.nanmean(vals))
    summary[f"{key}_std"]  = float(np.nanstd(vals, ddof=1)) if len(vals) > 1 else 0.0

df_seed_results = pl.DataFrame([
    {k: v for k, v in r.items() if k in ["seed", "best_epoch", "best_val_loss", *metrics_keys]}
    for r in seed_results
])
df_summary = pl.DataFrame([summary])

print("\nPer-seed test metrics:")
display(df_seed_results)
print("\nMean ± std across seeds:")
display(df_summary)

if best_result is None:
    raise RuntimeError("No valid seed result produced. Check training logs.")

# Expose best-seed model for downstream cells
model        = best_result["model"]
eval_model   = best_result["eval_model"]
test_dataset = best_result["test_dataset"]
print("\nBest seed:", best_result["seed"])


## Explanation Quality — B-COS Intrinsic

Explanation scores: `contrib['x'].abs().sum(dim=-1)` — the L1 norm of the node
feature contribution map for the **predicted class**.

Unlike binary classification (BA2Motif), no sign flip is needed here because
`explain_edge_attr` accepts a `target` class index explicitly, so the contribution
map is always oriented toward the correct output neuron.


In [ ]:
import networkx as nx

# ── Aggregate quantitative metrics (best-seed model) ─────────────────────────
avg_jaccard = evaluate_jaccard_edge(eval_model, test_dataset)
avg_auroc   = evaluate_auroc_edge(eval_model,   test_dataset)
print(f"Explanation Jaccard@|GT| (test): {avg_jaccard:.4f}")
print(f"Explanation Node AUROC   (test): {avg_auroc:.4f}")

# ── Single-graph qualitative explanation ─────────────────────────────────────
sample_idx = 0
data  = test_dataset[sample_idx].clone().to(DEVICE)
batch = torch.zeros(data.x.size(0), dtype=torch.long, device=DEVICE)

with torch.no_grad():
    logits     = eval_model(data.x, data.edge_index, data.edge_attr, batch)
pred_class = int(logits.argmax(dim=-1).item())

contrib = explain_edge_attr(
    eval_model, data.x, data.edge_index, data.edge_attr, batch, target=pred_class
)
# Node importance: L1 norm of the feature-wise contribution map (unsigned)
scores = contrib["x"].abs().sum(dim=-1).detach().cpu()

gt_mask = data.explanation_mask.squeeze().detach().cpu().numpy().astype(int)
k = int(gt_mask.sum())
topk = torch.topk(scores, k=k).indices if k > 0 else torch.tensor([], dtype=torch.long)

true_cls = int(data.y.item())
print(f"\nSample {sample_idx}")
print(f"  True class : {true_cls}  ({CLASS_LABELS[true_cls]})")
print(f"  Pred class : {pred_class} ({CLASS_LABELS[pred_class]})")
print(f"  GT motif nodes : {np.where(gt_mask)[0].tolist()}")
print(f"  Top-{k} explained: {topk.tolist()}")

# ── Visualization ─────────────────────────────────────────────────────────────
g = nx.Graph()
g.add_nodes_from(range(data.num_nodes))
g.add_edges_from(data.edge_index.cpu().t().tolist())
pos = nx.spring_layout(g, seed=42)

gt_nodes   = set(np.where(gt_mask)[0].tolist())
pred_nodes = set(topk.tolist())
node_colors = [
    "#2ca02c" if i in gt_nodes and i in pred_nodes   # TP — green
    else "#d62728" if i in gt_nodes                  # FN — red
    else "#ff7f0e" if i in pred_nodes                # FP — orange
    else "#aec7e8"                                    # TN — light blue
    for i in range(data.num_nodes)
]

plt.figure(figsize=(6, 6))
nx.draw(g, pos, with_labels=True, node_color=node_colors,
        node_size=380, font_color="white", edge_color="#666666")
plt.title(
    f"B-COS GINE explanation\n"
    f"True: {CLASS_LABELS[true_cls]}  |  Pred: {CLASS_LABELS[pred_class]}"
)
plt.show()


## Completeness Check

For multi-class GINE, completeness requires summing **both** node and edge
contributions for the predicted class logit:

$$f_c(x) \approx \sum_v M_c(x)_v + \sum_e M_c(e)_e$$

A low MAE confirms the B-COS dynamic-linear decomposition is exact.


In [ ]:
plt.rcParams.update({
    'font.size': 14, 'axes.labelsize': 16, 'axes.labelweight': 'bold',
    'axes.titlesize': 16, 'axes.titleweight': 'bold',
    'xtick.labelsize': 14, 'ytick.labelsize': 14, 'legend.fontsize': 14,
    'figure.figsize': (6, 6), 'axes.linewidth': 2.0,
})

bcos_logits       = []
bcos_contrib_sums = []

for data in tqdm(test_dataset, desc="Completeness Check"):
    data = data.clone().to(DEVICE)
    data.batch = torch.zeros(data.x.size(0), dtype=torch.long, device=DEVICE)

    with torch.no_grad():
        logits   = model(data.x, data.edge_index, data.edge_attr, data.batch)
        pred_cls = int(logits.argmax(dim=-1).item())
        # The logit for the predicted class is the completeness target
        tgt_logit = logits[0, pred_cls].item()

    # Signed contribution sums — no abs() here (completeness requires signed math)
    contrib    = explain_edge_attr(
        model, data.x, data.edge_index, data.edge_attr, data.batch, target=pred_cls
    )
    node_sum = contrib['x'].sum().item()
    edge_sum = contrib['edge_attr'].sum().item() if contrib['edge_attr'] is not None else 0.0

    bcos_logits.append(tgt_logit)
    bcos_contrib_sums.append(node_sum + edge_sum)

mae = np.mean(np.abs(np.array(bcos_logits) - np.array(bcos_contrib_sums)))
print(f"B-COS GINE Completeness Gap (MAE): {mae:.2e}")

fig, ax = plt.subplots()
ax.scatter(bcos_logits, bcos_contrib_sums, alpha=1.0, s=60,
           label=f'B-COS GINE (MAE: {mae:.2e})', color='navy',
           edgecolors='black', linewidths=0.5)
mn = min(min(bcos_logits), min(bcos_contrib_sums))
mx = max(max(bcos_logits), max(bcos_contrib_sums))
ax.plot([mn, mx], [mn, mx], color='#d62728', linestyle='--', linewidth=2.5,
        label='Perfect Completeness (y = x)')
ax.set_xlabel('Logit for Predicted Class  $f_c(x)$')
ax.set_ylabel('Sum of Contribution Map  $\\sum M_c(x)$')
legend = ax.legend(loc='upper left', frameon=True, edgecolor='black', framealpha=1.0)
for t in legend.get_texts():
    t.set_weight("bold")
ax.grid(True, linestyle='-', linewidth=0.5, alpha=0.7)
ax.tick_params(axis='both', which='major', width=2.0, length=6)
plt.tight_layout()
plt.savefig('completeness_check_dihalo.pdf', format='pdf', bbox_inches='tight')
plt.show()


## Qualitative Results (RDKit)

One example per class: ground-truth motif atoms (blue) vs. top-k B-COS explained atoms
(green = true positive, red = false positive).


In [ ]:
import io
import matplotlib.patches as mpatches
from PIL import Image
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.Draw import rdMolDraw2D

# ZINC atom feature index → atomic number
ATOM_MAP = {0: 6, 1: 7, 2: 8, 3: 16, 4: 9, 5: 17, 6: 35, 7: 53, 8: 15}


def pyg_to_rdkit(data):
    """Convert a PyG Data object to an RDKit molecule (single bonds, sanitized)."""
    mol = Chem.RWMol()
    for i in range(data.num_nodes):
        feat_idx = data.x[i].argmax().item() if data.x.dim() > 1 else int(data.x[i].item())
        mol.AddAtom(Chem.Atom(ATOM_MAP.get(feat_idx, 6)))
    ei = data.edge_index.cpu().numpy()
    for k in range(ei.shape[1]):
        u, v = int(ei[0, k]), int(ei[1, k])
        if u < v:
            mol.AddBond(u, v, Chem.BondType.SINGLE)
    try:
        Chem.SanitizeMol(mol)
    except Exception:
        pass
    return mol


def draw_rdkit_hd(mol, atom_colors=None):
    """Render molecule as a 600×600 HD PIL image with optional atom highlights."""
    d2d = rdMolDraw2D.MolDraw2DCairo(600, 600)
    opts = d2d.drawOptions()
    opts.minFontSize = 40
    opts.maxFontSize = 55
    opts.bondLineWidth = 5
    opts.padding = 0.05
    if not mol.GetNumConformers():
        AllChem.Compute2DCoords(mol)
    if atom_colors:
        d2d.DrawMolecule(mol, highlightAtoms=list(atom_colors.keys()),
                         highlightAtomColors=atom_colors)
    else:
        d2d.DrawMolecule(mol)
    d2d.FinishDrawing()
    return Image.open(io.BytesIO(d2d.GetDrawingText()))


def pick_one_per_class(dataset, num_classes, seed=42):
    """Return one test-graph index per class (strict: 8–32 nodes + has explanation_mask)."""
    candidates = {c: [] for c in range(num_classes)}
    rng = np.random.default_rng(seed)
    for idx in rng.permutation(len(dataset)):
        d = dataset[int(idx)]
        y = int(d.y.item())
        if hasattr(d, 'explanation_mask') and d.explanation_mask is not None:
            if 8 <= d.num_nodes <= 32:
                candidates[y].append(int(idx))
    return [candidates[c][0] if candidates[c] else 0 for c in range(num_classes)]


# ── Plotting ──────────────────────────────────────────────────────────────────
model.eval()
eval_model.eval()

selected = pick_one_per_class(test_dataset, num_classes, seed=42)
cols = len(selected)

fig, axes = plt.subplots(2, cols, figsize=(5 * cols, 12), dpi=200)
plt.subplots_adjust(wspace=0.02, hspace=0.05, left=0.06, right=0.98)

for col, idx in enumerate(selected):
    data = test_dataset[idx].clone().to(DEVICE)
    data.batch = torch.zeros(data.x.size(0), dtype=torch.long, device=DEVICE)
    true_cls = int(data.y.item())
    mol = pyg_to_rdkit(data)

    # ── Row 1: Ground truth (blue) ────────────────────────────────────────
    if hasattr(data, 'explanation_mask') and data.explanation_mask is not None:
        gt_mask  = data.explanation_mask.cpu().numpy().astype(bool)
        gt_atoms = set(int(i) for i in np.where(gt_mask)[0])
    else:
        gt_atoms = set()

    img_gt = draw_rdkit_hd(mol, {i: (0.2, 0.6, 1.0) for i in gt_atoms} or None)
    axes[0, col].imshow(img_gt)
    axes[0, col].set_title(CLASS_LABELS[true_cls], fontsize=11, fontweight='bold')
    axes[0, col].axis('off')

    # ── Row 2: B-COS prediction (green / red) ────────────────────────────
    with torch.no_grad():
        logits   = eval_model(data.x, data.edge_index, data.edge_attr, data.batch)
        pred_cls = int(logits.argmax(dim=-1).item())

    contrib = explain_edge_attr(
        eval_model, data.x, data.edge_index, data.edge_attr, data.batch, target=pred_cls
    )
    scores = contrib['x'].abs().sum(dim=-1).detach().cpu().numpy()
    k      = max(len(gt_atoms), 1)
    top_k  = set(np.argsort(scores)[-k:].tolist())

    inter = len(gt_atoms & top_k);  union = len(gt_atoms | top_k)
    jacc  = inter / union if union > 0 else 0.0

    pred_colors = {i: ((0.2, 0.8, 0.2) if i in gt_atoms else (0.9, 0.2, 0.2))
                   for i in top_k}
    img_pred = draw_rdkit_hd(mol, pred_colors or None)
    axes[1, col].imshow(img_pred)
    axes[1, col].set_title(f"Jaccard: {jacc:.2f}", fontsize=11, fontweight='bold')
    axes[1, col].text(0.5, 0.04, f"Pred: {CLASS_LABELS[pred_cls]}",
                      transform=axes[1, col].transAxes,
                      ha='center', fontsize=9, fontweight='bold',
                      bbox=dict(facecolor='white', alpha=0.9, edgecolor='gray',
                                boxstyle='round,pad=0.3'))
    axes[1, col].axis('off')

fig.text(0.01, 0.75, "Ground Truth", ha='left', va='center', rotation=90,
         fontsize=18, fontweight='bold')
fig.text(0.01, 0.25, "B-COS Explanation", ha='left', va='center', rotation=90,
         fontsize=18, fontweight='bold')

legend_elements = [
    mpatches.Patch(color=(0.2, 0.6, 1.0), label='Ground Truth Motif'),
    mpatches.Patch(color=(0.2, 0.8, 0.2), label='True Positive'),
    mpatches.Patch(color=(0.9, 0.2, 0.2), label='False Positive'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=3,
           bbox_to_anchor=(0.5, 0.0), frameon=True, fontsize=13,
           borderpad=1, edgecolor='gray')
plt.suptitle("B-COS GINE — Di-Halo Benzene Explanation Quality",
             fontsize=16, fontweight='bold', y=1.0)
plt.tight_layout(rect=[0.04, 0.08, 1.0, 0.97])
plt.savefig('dihalo_bcos_gine_explanation.pdf', format='pdf', bbox_inches='tight')
plt.show()


## Post-hoc Explainer Comparison on B-COS GINE

Applies **GNNExplainer** and **Integrated Gradients (IG)** as post-hoc methods to the
**same trained B-COS GINE model** and the **same test splits**, then compares all three
explanation methods side by side.

**Adapter note**: PyG's `Explainer` expects the model to accept `(x, edge_index, edge_attr, batch)`
as keyword arguments, which `EvalModelAdapter` already provides.  For IG, a 2-class adapter
is **not** needed here because the model already outputs multi-class logits `[N, num_classes]`.


In [ ]:
# ── Post-hoc explainer setup ──────────────────────────────────────────────────
from torch_geometric.explain import Explainer, GNNExplainer, ModelConfig

try:
    from torch_geometric.explain import CaptumExplainer
    CAPTUM_AVAILABLE = True
except ImportError:
    CAPTUM_AVAILABLE = False
    print("WARNING: CaptumExplainer not available. IG will be skipped.")


def make_gnn_explainer(model, gnn_epochs: int = 200, gnn_lr: float = 0.01):
    """GNNExplainer for a multi-class GINE.

    ``edge_mask_type='object'`` is set because our model uses edge_attr —
    PyG will also produce an edge mask alongside the node mask.
    """
    return Explainer(
        model=model,
        algorithm=GNNExplainer(epochs=gnn_epochs, lr=gnn_lr),
        explanation_type='model',
        node_mask_type='object',
        edge_mask_type='object',        # ← edge_attr is used by GINE
        model_config=ModelConfig(
            mode='multiclass_classification',
            task_level='graph',
            return_type='raw',
        ),
    )


def make_ig_explainer(model):
    """IG explainer for a multi-class GINE.

    Uses ``node_mask_type='attributes'`` so the node mask has shape
    ``[num_nodes, num_features]`` — required by Captum's path-integral.
    The B-COS model outputs ``[1, num_classes]``  directly (no wrapper needed).
    """
    if not CAPTUM_AVAILABLE:
        raise RuntimeError("Install captum (pip install captum) to use IG.")
    return Explainer(
        model=model,
        algorithm=CaptumExplainer('IntegratedGradients'),
        explanation_type='model',
        node_mask_type='attributes',
        edge_mask_type=None,
        model_config=ModelConfig(
            mode='multiclass_classification',
            task_level='graph',
            return_type='probs',
        ),
    )


def evaluate_ig_on_gine(ig_explainer, model, dataset, transform=None,
                        gt_attr: str = "explanation_mask"):
    """Jaccard@|GT| and Node AUROC for IG on the multi-class GINE.

    Node scores = abs(node_mask).sum(dim=1) — same approach as IG_BA2Motif.ipynb.
    """
    from sklearn.metrics import roc_auc_score as sk_auroc
    jaccards, aurocs = [], []
    device = next(model.parameters()).device

    for data in tqdm(dataset, desc="Evaluating IG (post-hoc)", leave=True):
        data_t  = transform(data.clone()) if transform else data.clone()
        data_t  = data_t.to(device)
        gt      = getattr(data_t, gt_attr, None)
        if gt is None:
            continue
        gt_mask = gt.squeeze().detach().cpu().numpy().astype(int)
        k       = int(gt_mask.sum())
        batch   = torch.zeros(data_t.x.size(0), dtype=torch.long, device=device)

        with torch.no_grad():
            logits    = model(data_t.x, data_t.edge_index, data_t.edge_attr, batch)
            pred_cls  = int(logits.argmax(dim=-1).item())

        try:
            expl   = ig_explainer(
                x=data_t.x, edge_index=data_t.edge_index,
                edge_attr=data_t.edge_attr, batch=batch, target=pred_cls
            )
            scores = expl.node_mask.abs().sum(dim=1).detach().cpu().numpy()

            if gt_mask.min() != gt_mask.max():
                aurocs.append(sk_auroc(gt_mask, scores))

            if k > 0:
                top_k    = set(np.argsort(scores)[-k:].tolist())
                gt_set   = set(np.where(gt_mask)[0].tolist())
                inter    = len(gt_set & top_k)
                union    = len(gt_set | top_k)
                jaccards.append(inter / union if union > 0 else 0.0)
        except Exception:
            pass

    return (
        float(np.mean(jaccards)) if jaccards else float('nan'),
        float(np.mean(aurocs))   if aurocs   else float('nan'),
    )


print("Post-hoc GINE explainer utilities ready.")
print(f"  Captum available: {CAPTUM_AVAILABLE}")


In [ ]:
# ── Per-seed post-hoc evaluation ─────────────────────────────────────────────
posthoc_results = []

for result in seed_results:
    seed_id = result["seed"]
    ev_m    = result["eval_model"].to(DEVICE)
    t_d     = result["test_dataset"]

    print(f"\n{'='*60}")
    print(f"Seed {seed_id}: post-hoc explainers on B-COS GINE")
    print(f"{'='*60}")

    ev_m.eval()

    # ── GNNExplainer ──────────────────────────────────────────────────────
    gnn_exp   = make_gnn_explainer(ev_m, gnn_epochs=200, gnn_lr=0.01)
    gnn_jacc  = evaluate_gnnexplainer_jaccard_edge(gnn_exp, ev_m, t_d)
    gnn_auroc = evaluate_gnnexplainer_auroc_edge(gnn_exp,   ev_m, t_d)
    print(f"  GNNExplainer  →  Jaccard: {gnn_jacc:.4f}  |  AUROC: {gnn_auroc:.4f}")

    # ── Integrated Gradients ──────────────────────────────────────────────
    if CAPTUM_AVAILABLE:
        ig_exp = make_ig_explainer(ev_m)
        ig_jacc, ig_auroc = evaluate_ig_on_gine(ig_exp, ev_m, t_d)
        print(f"  IG (Captum)   →  Jaccard: {ig_jacc:.4f}  |  AUROC: {ig_auroc:.4f}")
    else:
        ig_jacc = ig_auroc = float('nan')
        print("  IG (Captum)   →  SKIPPED (captum not installed)")

    posthoc_results.append({
        "seed":        seed_id,
        "gnn_jaccard": float(gnn_jacc),
        "gnn_auroc":   float(gnn_auroc),
        "ig_jaccard":  float(ig_jacc),
        "ig_auroc":    float(ig_auroc),
    })

print("\nPost-hoc evaluation complete.")


In [ ]:
# ── Comparison summary table ──────────────────────────────────────────────────
def _ms(vals):
    """Mean ± std string, ignoring NaN values."""
    vals = np.asarray([v for v in vals if not np.isnan(v)], dtype=float)
    if len(vals) == 0:
        return "N/A"
    std = vals.std(ddof=1) if len(vals) > 1 else 0.0
    return f"{vals.mean():.4f} ± {std:.4f}"


bcos_jacc  = [r["test_jaccard"]    for r in seed_results]
bcos_auroc = [r["test_expl_auroc"] for r in seed_results]
gnn_jacc   = [r["gnn_jaccard"]     for r in posthoc_results]
gnn_auroc  = [r["gnn_auroc"]       for r in posthoc_results]
ig_jacc    = [r["ig_jaccard"]      for r in posthoc_results]
ig_auroc   = [r["ig_auroc"]        for r in posthoc_results]

comparison_df = pl.DataFrame([
    {
        "Method":       "B-COS Intrinsic",
        "Model":        "B-COS GINE",
        "Explainer":    "Linear decomposition (explain_edge_attr.py)",
        "Jaccard@|GT|": _ms(bcos_jacc),
        "Node AUROC":   _ms(bcos_auroc),
    },
    {
        "Method":       "GNNExplainer (post-hoc)",
        "Model":        "B-COS GINE",
        "Explainer":    "PyG GNNExplainer (200 epochs)",
        "Jaccard@|GT|": _ms(gnn_jacc),
        "Node AUROC":   _ms(gnn_auroc),
    },
    {
        "Method":       "IG (post-hoc)",
        "Model":        "B-COS GINE",
        "Explainer":    "Captum IntegratedGradients",
        "Jaccard@|GT|": _ms(ig_jacc),
        "Node AUROC":   _ms(ig_auroc),
    },
])

print("=" * 75)
print("Explanation Quality Comparison — Di-Halo Benzene (B-COS GINE Model)")
print(f"Across {len(seed_results)} seeds, mean ± std")
print("=" * 75)
display(comparison_df)
